# Pipeline Preditivo de Vendas - Varejo B2B/B2C (Arquitetura V2.2)

Este notebook documenta, de forma acadêmica e reproduzível, o desenvolvimento do modelo de *Machine Learning* vencedor para previsão de demanda no varejo (solução do Hackathon 2025). O objetivo é prever a quantidade de vendas semanais de produtos (`sku`) em diferentes pontos de venda (`pdv`) para as primeiras semanas de 2023, utilizando dados históricos de 2022.

## 1. Racional Arquitetural e Decisões de Design
Durante a fase experimental, diversas abordagens foram exaustivamente testadas. As decisões arquiteturais consolidadas nesta versão (V2.2) são:

- **O Algoritmo - Por que LightGBM?** A base de dados contém 5.6 milhões de registros semanais cruzados com 10 variáveis dimensionais estáticas de altíssima cardinalidade (ex: `fabricante` com 343 valores únicos, `pdv` com centenas). Testes com `CatBoost` demonstraram lentidão severa e alto custo computacional (consumo superior a 23 GB de RAM por mais de 10 horas em CPU) devido às permutações para o *Target Encoding*. O `LightGBM` provou-se formidável devido ao seu *Histogram-based Splitting*, capaz de tratar features categóricas de forma otimizada e nativa, reduzindo o tempo de treinamento para ~15 minutos, sem perda de acurácia.
- **O Experimento Frustrado da Transformação log1p:** A distribuição da variável alvo (vendas) é extremamente assimétrica (mediana 2, mas picos superiores a 90.000). Uma abordagem padrão seria aplicar o logaritmo (`np.log1p(y)`) para suavizar o gradiente. No entanto, o logaritmo minimiza essencialmente o *Erro Percentual* (MAPE). Ao reconverter a previsão para a escala linear (`np.expm1`), observou-se uma degradação severa no *Erro Absoluto* (a métrica oficial da competição é o MAE). Dessa forma, optou-se por treinar o algoritmo estritamente na escala original.
- **AutoML:** Embora frameworks de AutoML (como o AutoGluon) pudessem espremer décimos extras no MAE usando ensacamento e *Stacking* profundo (redes neurais + xgboost + catboost), eles esbarram no mesmo gargalo computacional e falham na viabilidade produtiva (escalabilidade) de um ambiente sem processamento massivo de GPUs. Optou-se então por um LightGBM altamente otimizado por buscas Bayesianas.

In [ ]:
# Importando bibliotecas necessárias
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error
from optuna.integration import LightGBMPruningCallback

# Configurando estilos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Carregamento e Enriquecimento de Dados

A modelagem preditiva no varejo se beneficia exponencialmente do enriquecimento dos dados. Neste bloco, cruzamos os dados transacionais de vendas (`vendas.parquet`) com as tabelas dimensionais de ponto de venda (`pdv.parquet`) e hierarquia de produtos (`produtos.parquet`).

**Agregação Semanal:** O ruído diário das vendas (dias da semana sem estoque ou sem movimento) prejudica o aprendizado. Ao agregar as vendas por `ano`, `semana`, `pdv` e `sku`, conseguimos uma curva temporal muito mais suave e passível de predição, criando a variável `preco_medio_unitario` baseada na proporção de receita / volume.

In [ ]:
def carregar_e_preparar_dados():
    print("Carregando datasets brutos...")
    df_vendas = pd.read_parquet('data/raw/vendas.parquet')
    df_pdvs = pd.read_parquet('data/raw/pdv.parquet')
    df_produtos = pd.read_parquet('data/raw/produtos.parquet')
    
    print("Realizando junção e enriquecimento...")
    df_merged = pd.merge(df_vendas, df_pdvs, left_on='internal_store_id', right_on='pdv', how='inner')
    df_merged = pd.merge(df_merged, df_produtos, left_on='internal_product_id', right_on='produto', how='inner')

    # Convertendo datas para calendário semanal ISO
    df_merged['transaction_date'] = pd.to_datetime(df_merged['transaction_date'])
    df_merged['ano'] = df_merged['transaction_date'].dt.isocalendar().year
    df_merged['semana'] = df_merged['transaction_date'].dt.isocalendar().week

    dim_cols = ['categoria_pdv', 'premise', 'categoria', 'subcategoria', 'tipos', 'label', 'marca', 'fabricante']
    group_cols = ['ano', 'semana', 'pdv', 'produto'] + dim_cols

    print("Agregando vendas temporalmente...")
    agg_vendas = df_merged.groupby(group_cols).agg(
        total_quantity=('quantity', 'sum'),
        total_gross_value=('gross_value', 'sum'),
    ).reset_index()

    agg_vendas = agg_vendas.rename(columns={'produto': 'sku', 'total_quantity': 'quantidade'})
    
    # O preço unitário capta o posicionamento e elasticidade
    agg_vendas['preco_medio_unitario'] = np.where(
        agg_vendas['quantidade'] > 0,
        agg_vendas['total_gross_value'] / agg_vendas['quantidade'],
        0.0
    )
    agg_vendas.drop(columns=['total_gross_value'], inplace=True)
    print(f"Base preparada: {agg_vendas.shape[0]} linhas x {agg_vendas.shape[1]} colunas.")
    return agg_vendas

# Descomente a linha abaixo para carregar os dados se tiver os parquets na pasta data/raw
# df_base = carregar_e_preparar_dados()

## 3. Feature Engineering Avançada (V2.2)

A base de conhecimento do algoritmo deriva fundamentalmente de 32 features temporais desenhadas para traduzir o conceito matemático de **Time Series** em features tabulares autoexplicativas:

1. **Defasagens Temporais (Lags):** Foram criados lags de vendas (t-1, t-2, t-3, t-4, t-12, t-52). Isso captura tanto correlações lineares curtas (semana passada) quanto sazonalidades severas (mesma época do ano passado).
2. **Médias e Variâncias Móveis (Rolling Windows):** Janelas de 4, 12 e 52 semanas indicam ao modelo o platô de tendência do produto, amortecendo a volatilidade. O Coeficiente de Variação ($ \sigma / \mu $) indica estabilidade (ou a falta dela).
3. **Ciclos Harmônicos:** Seno e cosseno da semana capturam a ciclicidade contínua do ano, fechando a transição temporal entre a semana 52 e a semana 1.
4. **Tendência Implícita e Preço:** O delta (diferença) entre o lag 1 e lag 2 sinaliza aceleração nas vendas (momentum). O lag temporal do preço unitário ajuda a modelar a elasticidade da demanda.

In [ ]:
def criar_features(df):
    print("Engenharia de features temporais e estatísticas...")
    df_feat = df.copy()
    df_feat.sort_values(['pdv', 'sku', 'ano', 'semana'], inplace=True)
    df_feat.reset_index(drop=True, inplace=True)

    # Calendário e Ciclos
    df_feat['trimestre'] = (df_feat['semana'] - 1) // 13 + 1
    df_feat['seno_semana'] = np.sin(2 * np.pi * df_feat['semana'] / 52)
    df_feat['cosseno_semana'] = np.cos(2 * np.pi * df_feat['semana'] / 52)

    # Lags de Quantidade
    grouped_qty = df_feat.groupby(['pdv', 'sku'])['quantidade']
    lags = [1, 2, 3, 4, 12, 52]
    for lag in lags:
        df_feat[f'lag_{lag}_semanas'] = grouped_qty.shift(lag)

    # Lags de Preço e Tendência (Momentum)
    df_feat['lag_1_preco'] = df_feat.groupby(['pdv', 'sku'])['preco_medio_unitario'].shift(1)
    df_feat['lag_diff_1'] = df_feat['lag_1_semanas'] - df_feat['lag_2_semanas']

    # Janelas Estatísticas Móveis (Rolling)
    shifted = grouped_qty.shift(1)
    tmp = pd.DataFrame({'val': shifted, 'pdv': df_feat['pdv'], 'sku': df_feat['sku']})
    tmp_grouped = tmp.groupby(['pdv', 'sku'])['val']
    
    for window in [4, 12, 52]:
        roll = tmp_grouped.rolling(window=window, min_periods=1)
        df_feat[f'rolling_mean_{window}_semanas'] = roll.mean().reset_index(level=[0, 1], drop=True)
        df_feat[f'rolling_std_{window}_semanas'] = roll.std().reset_index(level=[0, 1], drop=True)
        df_feat[f'rolling_max_{window}_semanas'] = roll.max().reset_index(level=[0, 1], drop=True)
        if window in [4, 12]:
            df_feat[f'rolling_min_{window}_semanas'] = roll.min().reset_index(level=[0, 1], drop=True)

    # Coeficiente de Variação
    df_feat['coef_variacao_4'] = np.where(df_feat['rolling_mean_4_semanas'] > 0,
                                          df_feat['rolling_std_4_semanas'] / df_feat['rolling_mean_4_semanas'], 0.0)
    
    df_feat.fillna(0, inplace=True)
    return df_feat

# df_features = criar_features(df_base)

## 4. Otimização Bayesiana com Optuna

A hiperparametrização é crítica para o LightGBM não sobreajustar (overfitting) o ruído dos dados de varejo. O Optuna aplica Processos Gaussianos ou TPE (*Tree-structured Parzen Estimator*) para navegar inteligentemente pelo espaço de busca de parâmetros. 
Nossa configuração incorpora um **Early Pruning (Poda)**: se nos primeiros *warmup_steps* as métricas de validação cruzada do trial demonstrarem ser significativamente piores do que a média (usando `MedianPruner`), o trial é abortado precocemente, economizando imenso tempo computacional (em testes ~20 dos 30 trials são podados sem afetar o resultado ótimo).

*O conjunto de Validação é um OOT (Out-of-Time) hold-out das semanas 48 a 52.*

In [ ]:
categorical_features = ['pdv', 'sku', 'categoria_pdv', 'premise', 
                        'categoria', 'subcategoria', 'tipos', 'label', 'marca', 'fabricante']
                        
feature_names = ['semana', 'trimestre', 'seno_semana', 'cosseno_semana'] + categorical_features + \
                ['lag_1_semanas', 'lag_2_semanas', 'lag_3_semanas', 'lag_4_semanas', 'lag_12_semanas', 'lag_52_semanas'] + \
                ['lag_1_preco', 'lag_diff_1', 'coef_variacao_4', 'preco_medio_unitario'] + \
                ['rolling_mean_4_semanas', 'rolling_std_4_semanas', 'rolling_max_4_semanas', 'rolling_min_4_semanas'] + \
                ['rolling_mean_12_semanas', 'rolling_std_12_semanas', 'rolling_max_12_semanas', 'rolling_min_12_semanas'] + \
                ['rolling_mean_52_semanas', 'rolling_std_52_semanas', 'rolling_max_52_semanas']

def treinar_modelo_optuna(df_feat, n_trials=30):
    # Converter categoricas para o tipo Category exigido nativamente pelo pandas/lightgbm
    for col in categorical_features:
        df_feat[col] = df_feat[col].astype('category')

    train_set = df_feat[df_feat['semana'] < 48]
    val_set = df_feat[df_feat['semana'] >= 48]

    X_train, y_train = train_set[feature_names], train_set['quantidade']
    X_val, y_val = val_set[feature_names], val_set['quantidade']

    def objective(trial):
        params = {
            'objective': 'regression_l1', # MAE como função alvo e metrica
            'metric': 'mae',
            'verbosity': -1,
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 256),
            'max_depth': trial.suggest_int('max_depth', 5, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
            'random_state': 42,
            'n_jobs': -1
        }
        pruning_callback = LightGBMPruningCallback(trial, "l1")
        model = lgb.LGBMRegressor(**params)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(20, verbose=False), pruning_callback],
            categorical_feature=categorical_features
        )
        preds = model.predict(X_val)
        return mean_absolute_error(y_val, preds)

    print(f"Iniciando Busca Bayesiana com {n_trials} trials...")
    study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"Melhor MAE alcançado: {study.best_value:.4f}")
    print(f"Hiperparâmetros Ótimos: {study.best_params}")
    
    # Treinar modelo final robusto
    final_params = study.best_params
    final_params['n_estimators'] = 1000  # Expandimos árvores e controlamos com early-stopping
    model_final = lgb.LGBMRegressor(objective='regression_l1', random_state=42, verbosity=-1, **final_params)
    
    model_final.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)], eval_metric="mae",
        callbacks=[lgb.early_stopping(50, verbose=False)],
        categorical_feature=categorical_features
    )
    return model_final

# Descomente a linha abaixo para realizar o treinamento
# model_final = treinar_modelo_optuna(df_features, n_trials=100)

## 5. Inferência Autorregressiva (Previsões de 2023)

Em problemas temporais contínuos, prever o futuro distante sem atualizar as referências passadas destrói a precisão. Como prever a semana 5 se não sabemos o real valor que ocorreu na semana 4?

**Solução:** Abordagem Autorregressiva. Calculamos a semana 1 usando dados históricos reais. Em seguida, pegamos as predições de vendas dessa semana, e *as injetamos temporariamente de volta no conjunto de dados* como se fossem o passado. Refazemos toda a Engenharia de Features (os Lags de 1 semana agora enxergam a previsão) e, baseados nessa projeção matemática, projetamos a semana 2. E assim sucessivamente.

In [ ]:
def gerar_submissao(model, df_hist, weeks_to_forecast=5):
    forecast_df = df_hist.copy()
    all_forecasts = []

    for current_week in range(1, weeks_to_forecast + 1):
        print(f"Previsão Autorregressiva: Processando Semana {current_week}/2023...")
        # Atualiza engenharia com o novo passado projetado
        features_base = criar_features(forecast_df)
        latest_entries = features_base.sort_values(by=['ano', 'semana']).drop_duplicates(subset=['pdv', 'sku'], keep='last')
        
        X_pred = latest_entries.copy()
        X_pred['semana'] = current_week
        X_pred['ano'] = 2023

        # Harmoniza as classes categóricas no DataFrame de predição
        for col in categorical_features:
            idx = categorical_features.index(col)
            model_categories = model.booster_.pandas_categorical[idx]
            X_pred[col] = pd.Categorical(X_pred[col], categories=model_categories)

        predictions_raw = model.predict(X_pred[feature_names])
        predictions = np.maximum(0, np.round(predictions_raw)).astype(int)
        
        week_forecast = X_pred[['pdv', 'sku']].copy()
        week_forecast['semana'] = current_week
        week_forecast['quantidade_prevista'] = predictions
        all_forecasts.append(week_forecast)

        # Injeta as projeções no histórico para servir de base para o LAG 1 da próxima semana
        new_data = week_forecast.rename(columns={'quantidade_prevista': 'quantidade'})
        new_data['ano'] = 2023
        
        dim_cols_to_copy = [c for c in forecast_df.columns if c not in ['ano', 'semana', 'pdv', 'sku', 'quantidade', 'preco_medio_unitario']]
        for col in dim_cols_to_copy:
            if col in X_pred.columns:
                new_data[col] = X_pred[col].values
        
        new_data['preco_medio_unitario'] = X_pred['lag_1_preco'].values if 'lag_1_preco' in X_pred.columns else 0.0
        forecast_df = pd.concat([forecast_df, new_data], ignore_index=True)

    df_final = pd.concat(all_forecasts, ignore_index=True)
    return df_final

# df_submission = gerar_submissao(model_final, df_features)
# df_submission.to_csv('Predictive_Sales_Final.csv', index=False)
# print("Sucesso! Arquivo de submissão gerado.")